# Fig. S5: Squeezing spectrum of the seeded system (effective model)

Effective lasing model (cooling atoms and mode $b$ adiabatically eliminated: the squeezed mode $a_s$ decays at rate $\kappa_f$ and is shifted by $\Delta$) with the same seed, parameters and procedure as `1_full_model_seeded.ipynb`. Besides the squeezing spectrum we save the frequencies of the modes of the correlation-function equations (black lines of Fig. S5).

**Outputs** (`../data/`): `Ssq_eff.npy`, `eff_positions.npy`.

In [ ]:
using QuantumCumulants
using ModelingToolkit
using OrdinaryDiffEq
using WignerSymbols
using LinearAlgebra
using ProgressMeter
using NPZ
using QuantumOptics.timecorrelations: correlation2spectrum

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)
    
    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_k = 100

# Rb87 level structure (B in G; angular frequencies, with the frequency in GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

# Number of cooling (N_atom_I = N_c) and gain (N_atom_II = N_g) atoms
N_atom_I = 5000
N_atom_II = 300
N_atom = N_atom_I .+ N_atom_II

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = logrange(0.01,20,num_k)*2*pi*1e-3 # Cavity decay, scanned
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = 0.8 # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$ (steps 1–7), and the effective pumping, decay and dephasing rates of the atoms (step 8).

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav+3*w_r-w_s).-sqrt.(4*gr^2*N_atom.+(w_cav-w_r)^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom_I.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom_I*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

dec_eff = gamma*dipole_me(1,1,-1,0)^2*alfa.^2
P_eff = gamma*dipole_me(2,2,-2,0)^2*delta.^2
dep_eff = (gamma*dipole_me(1,2,-1,-1)^2*delta.^2 .+ gamma*dipole_me(2,1,-2,+1)^2*alfa.^2)/4
dec_eff_b = gamma_D2*(5/12)*alfa_b.^2
dep_eff_b = gamma_D2*(1/10)*alfa_b.^2/4

# Step 6: Eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom_I.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # coefficient of s's term

# Step 7: Eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom_I).^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom_I)./((delta_lamb.^2).+(N_atom_I.*dec/2).^2)

# Step 8: include pumping and dephasing

dec_T = dec_eff
dep_T = dep_eff

P = 1e-4
P_T = P.+P_eff

C = (2*N_atom_II.*g_02)./(((P_T.+dec_T)/2 .+ 2*dep_T).*(dec_f.+kappa))./(1 .+(4*delta_lamb_f.^2)./((kappa.+dec_f).+(P_T.+dec_T.+4*dep_T)).^2)
n_q = (P_T.-dec_T)./(2*(kappa.+dec_f)).*N_atom_II
control = P_T./(kappa.+dec_f).-((P_T.+dec_T)./(2*(kappa.+dec_f))).*(1 .+ 1 ./ C)

purc = 4*g_02./(dec_f.+kappa)

## Cumulant expansion of the effective model

In [ ]:
@rnumbers Δ λ_r λ_s κ κ_f R N p γ_ϕ γ Λ # Parameters

h_cav = FockSpace(:cavity) # Hilbert space of the cavity
h_atom = NLevelSpace(:atom,(:g,:e)) # Hilbert space of the atom
h = h_cav⊗h_atom # Complete Hilbert space

as = Destroy(h,:as)
a = as*cosh(R)-as'*sinh(R)
σ(x,y,k) = IndexedOperator(Transition(h, :σ, x, y), k) # Operators

i = Index(h, :i, N, h_atom)
j = Index(h, :j, N, h_atom)

H = -Δ*as'*as + ∑(λ_r*(a*σ(:e,:g,i)+a'*σ(:g,:e,i)) + λ_s*(a'*σ(:e,:g,i)+a*σ(:g,:e,i)),i) + Λ*im*(a-a')

D = [a;as;σ(:e,:g,i);σ(:e,:e,i)-σ(:g,:g,i);σ(:g,:e,i)]
rates = [κ; κ_f; p; γ_ϕ; γ]

ops = [as] # Operators whose equations we derive
eqs = meanfield(ops,H,D;rates=rates,order=2) # Second-order cumulant expansion
me_comp = complete!(eqs) # Complete the set of equations
eqs_sc = scale(me_comp) # Use the permutation symmetry of the atoms

corr = CorrelationFunction(as',as,me_comp; steady_state=true)
corr2 = CorrelationFunction(as,as',me_comp; steady_state=true)

corr_sc = scale(corr)
corr_sc2 = scale(corr2)

param_list = (Δ, λ_r, λ_s, κ, κ_f, R, N, p, γ_ϕ, γ, Λ)

@named sys = System(eqs_sc)
@named csys = System(corr_sc)
@named csys2 = System(corr_sc2)

## Squeezing spectrum

For each $\kappa$: steady state with the seed $\Lambda = 10^{-2}$, then the two-time correlation functions $\langle a_s^\dagger(\tau) a_s\rangle$, $\langle a_s(\tau) a_s\rangle$, $\langle a_s(\tau) a_s^\dagger\rangle$, $\langle a_s^\dagger(\tau) a_s^\dagger\rangle$ (coherent part subtracted) are integrated up to $\tau = 10^6$ with step $\Delta\tau = 2$ and Fourier-transformed. This gives a frequency resolution of 1 kHz over ±250 MHz; only $|\nu| \le 2$ MHz is saved.

In [ ]:
coh = zeros(ComplexF64,num_k) # <a_s>

Δτ = 2
τ = range(0.0, 1e6, step = Δτ) # Delay times of the correlation functions

freqs = zeros(length(τ))
spec = zeros(num_k,length(τ)) # Squeezing spectrum
emis_spec = zeros(num_k,length(τ)) # Emission spectrum

eigs = zeros(ComplexF64,num_k,length(corr_sc.de)) # Eigenvalues of the correlation-function equations

progress = Progress(num_k)
safe_lock = ReentrantLock()

Threads.@threads for l in 1:num_k

    # Steady state with the seed

    u0 = zeros(ComplexF64, length(eqs_sc))

    p0 = (delta_lamb_f, lambda_r, lambda_s, kappa[l], dec_f, r, N_atom_II, P_T, dep_T, dec_T , 1e-2)
    dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

    prob = ODEProblem(sys,dict,(0.0,1000000.0))
    sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

    lock(safe_lock) do
        coh[l] = sol[as][end]
    end

    # Evolution of correlator 1

    u0_c = lock(safe_lock) do
        correlation_u0(corr_sc, sol.u[end])
    end
    p0_c = lock(safe_lock) do
        correlation_p0(corr_sc, sol.u[end], param_list .=> p0)
    end
    dict_c = merge(Dict(u0_c), Dict(p0_c))

    prob_c = ODEProblem(csys,dict_c,(τ[1],τ[end]))
    sol_c = solve(prob_c, Tsit5(), save_idxs = [1,4], saveat=τ; abstol=1e-8, reltol=1e-8, maxiters=1e6)

    corr_ada = [sol_c.u[m][1] for m in 1:length(τ)] .- abs2(coh[l])
    corr_aa = [sol_c.u[m][2] for m in 1:length(τ)] .- coh[l]^2

    freqs_ada, s_ada = correlation2spectrum(τ, corr_ada)
    freqs_aa, s_aa = correlation2spectrum(τ, corr_aa)

    s_ada .-= Δτ*real(corr_ada[1])
    s_aa  .-= Δτ*real(corr_aa[1])

    # Evolution of correlator 2

    u0_c = lock(safe_lock) do
        correlation_u0(corr_sc2, sol.u[end])
    end
    p0_c = lock(safe_lock) do
        correlation_p0(corr_sc2, sol.u[end], param_list .=> p0)
    end
    dict_c = merge(Dict(u0_c), Dict(p0_c))

    prob_c = ODEProblem(csys2,dict_c,(τ[1],τ[end]))
    sol_c = solve(prob_c, Tsit5(), save_idxs = [1,4], saveat=τ; abstol=1e-8, reltol=1e-8, maxiters=1e6)

    corr_aad = [sol_c.u[m][1] for m in 1:length(τ)] .- abs2(coh[l])
    corr_adad = [sol_c.u[m][2] for m in 1:length(τ)] .- conj(coh[l])^2

    freqs_aad, s_aad = correlation2spectrum(τ, corr_aad)
    freqs_adad, s_adad = correlation2spectrum(τ, corr_adad)

    s_aad .-= Δτ*real(corr_aad[1])
    s_adad  .-= Δτ*real(corr_adad[1])

    # Spectra

    freqs[:] = freqs_ada
    emis_spec[l,:] = cosh(r)^2*s_ada .+ sinh(r)^2*s_aad.-cosh(r)*sinh(r)*(s_aa .+ s_adad)
    anom_spec = cosh(r)^2*s_aa .+ sinh(r)^2*s_adad.-cosh(r)*sinh(r)*(s_ada .+ s_aad)

    spec[l,:] = real.(1 .+ kappa[l]*(emis_spec[l,:] .+ reverse(emis_spec[l,:]) .+ anom_spec*exp(2*im*0) .+ reverse(anom_spec)*exp(-2*im*0)))

    # Eigenvalues of the correlation-function equations (Jacobian by finite differences)

    f(u) = (du = similar(u); prob_c.f(du, u, prob_c.p, 0.0); du)
    n  = length(prob_c.u0)
    f0 = f(zeros(ComplexF64, n))
    J  = reduce(hcat, [f(ComplexF64.(1:n .== k)) .- f0 for k in 1:n])
    D  = eigen(J)
    perm = sortperm(imag.(D.values), rev = true)
    eigs[l, :] = D.values[perm]

    next!(progress)
end

## Save

In [ ]:
sq_db = 20*log10.(sqrt.(spec)) # 10 log10(S) in dB: negative values mean squeezing

window = abs.(freqs/(2pi*1e-3)) .<= 2 + 1e-9 # Saved frequency window, |ν| ≤ 2 MHz

data_dir = joinpath("..", "data") # Relative to this notebook

npzwrite(joinpath(data_dir, "Ssq_eff.npy"), sq_db[:,window])
npzwrite(joinpath(data_dir, "eff_positions.npy"), imag.(eigs))